# Video Compression Using Higher-Order Singular Value Decomposition (HOSVD)

This notebook implements video compression using tensor decomposition. The input video is represented as a third-order tensor, decomposed using SVD along its three modes, and reconstructed using a reduced number of components. The compression level is controlled by the parameter k.

## 1. Install and Import Libraries

Install TensorLy and import the libraries required for video processing, tensor operations, and numerical computation.

In [ ]:
!pip install -q tensorly

import cv2
import tensorly as tl
import numpy as np
import matplotlib.pyplot as plt
import os

from google.colab import files

## 2. Upload the Input Video

Upload the video file to Google Colab. The default input filename used in this notebook is run.mp4.

In [ ]:
uploaded = files.upload()

video_path = 'run.mp4'

if video_path not in uploaded and not os.path.exists(video_path):
    raise FileNotFoundError('Please upload a video named run.mp4 or update video_path.')

print('Input video:', video_path)

## 3. Read Video Frames

Read all frames from the input video using OpenCV. Each frame is converted to grayscale so that the video can be represented as a third-order tensor with dimensions (number of frames, height, width). This also avoids storing three separate color channels.

In [ ]:
source = cv2.VideoCapture(video_path)

if not source.isOpened():
    raise ValueError('Could not open the input video.')

video_frame_count = int(source.get(cv2.CAP_PROP_FRAME_COUNT))
fps = source.get(cv2.CAP_PROP_FPS)
frame_width = int(source.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(source.get(cv2.CAP_PROP_FRAME_HEIGHT))

frames_array = []

while True:
    ret, frame = source.read()

    if not ret:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frames_array.append(gray_frame)

source.release()

if len(frames_array) == 0:
    raise ValueError('No frames were read from the input video.')

frames_array = np.asarray(frames_array, dtype=np.float32)

print('Reported frame count:', video_frame_count)
print('Read frame count:', len(frames_array))
print('Frame rate:', fps)
print('Frame dimensions:', frame_width, 'x', frame_height)
print('Frames array shape:', frames_array.shape)

## 4. Construct the Video Tensor

Convert the sequence of frames into a third-order tensor. Its dimensions correspond to the temporal, vertical, and horizontal modes:

\[T \in \mathbb{R}^{N \times H \times W}\]

where N is the number of frames, H is the frame height, and W is the frame width.

In [ ]:
vid_tensor = tl.tensor(frames_array)
vid_shape = vid_tensor.shape

print('Video tensor shape:', vid_shape)
print('Number of frames:', vid_shape[0])
print('Frame height:', vid_shape[1])
print('Frame width:', vid_shape[2])

## 5. Visualize Original Video Frames

Display selected frames from the original video to inspect the input before compression.

In [ ]:
frame_indices = np.linspace(0, vid_shape[0] - 1, min(6, vid_shape[0]), dtype=int)

fig, axes = plt.subplots(1, len(frame_indices), figsize=(16, 4))

if len(frame_indices) == 1:
    axes = [axes]

for ax, idx in zip(axes, frame_indices):
    ax.imshow(vid_tensor[idx], cmap='gray')
    ax.set_title(f'Frame {idx}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Tensor Unfolding

Unfold the video tensor along each of its three modes. Each unfolding operation converts the tensor into a matrix so that SVD can be applied.

In [ ]:
unfold1 = tl.unfold(vid_tensor, mode=0)
unfold2 = tl.unfold(vid_tensor, mode=1)
unfold3 = tl.unfold(vid_tensor, mode=2)

print('Mode-0 unfolding shape:', unfold1.shape)
print('Mode-1 unfolding shape:', unfold2.shape)
print('Mode-2 unfolding shape:', unfold3.shape)

## 7. SVD of the Unfolded Matrices

Perform Singular Value Decomposition on each unfolding matrix:

\[X_{(n)} = U_n \Sigma_n V_n^T\]

The left singular vectors contain the basis vectors for each tensor mode. Reduced SVD is used to obtain the necessary factor matrices efficiently.

In [ ]:
U1, s1, V1 = np.linalg.svd(unfold1, full_matrices=False)
U2, s2, V2 = np.linalg.svd(unfold2, full_matrices=False)
U3, s3, V3 = np.linalg.svd(unfold3, full_matrices=False)

print('U1 shape:', U1.shape)
print('U2 shape:', U2.shape)
print('U3 shape:', U3.shape)

print('\nNumber of singular values:')
print('Mode 0:', len(s1))
print('Mode 1:', len(s2))
print('Mode 2:', len(s3))

## 8. Compute the Core Tensor

The core tensor is obtained by multiplying the original video tensor by the transpose of each factor matrix along its corresponding mode:

\[S = T \times_1 U_1^T \times_2 U_2^T \times_3 U_3^T\]

The core tensor contains the information required to reconstruct the video from reduced factor matrices.

In [ ]:
A1 = tl.tenalg.mode_dot(vid_tensor, U1.T, mode=0)
A2 = tl.tenalg.mode_dot(A1, U2.T, mode=1)
S = tl.tenalg.mode_dot(A2, U3.T, mode=2)

print('Core tensor shape:', S.shape)

## 9. Truncate the Core Tensor

Choose the number of retained components in each mode. The parameter k controls the compression level. A smaller k results in stronger compression but may reduce reconstruction quality.

In [ ]:
k = 60

if k > min(vid_shape):
    raise ValueError(
        f'k={k} is too large for the video dimensions {vid_shape}. '
        f'Choose k <= {min(vid_shape)}.'
    )

S_reduced = S[:k, :k, :k]

U1_reduced = U1[:, :k]
U2_reduced = U2[:, :k]
U3_reduced = U3[:, :k]

print('Reduced core tensor shape:', S_reduced.shape)
print('Reduced U1 shape:', U1_reduced.shape)
print('Reduced U2 shape:', U2_reduced.shape)
print('Reduced U3 shape:', U3_reduced.shape)

## 10. Reconstruct the Compressed Video

Reconstruct the video by multiplying the reduced core tensor by the truncated factor matrices along all three modes:

\[\hat{T} = S_k \times_1 U_{1,k} \times_2 U_{2,k} \times_3 U_{3,k}\]

The reconstructed tensor has the same dimensions as the original video.

In [ ]:
T_reconst = tl.tenalg.mode_dot(S_reduced, U1_reduced, mode=0)
T_reconst = tl.tenalg.mode_dot(T_reconst, U2_reduced, mode=1)
T_reconst = tl.tenalg.mode_dot(T_reconst, U3_reduced, mode=2)

T_reconst = np.asarray(T_reconst)

print('Reconstructed video shape:', T_reconst.shape)

# Clip pixel values and convert to 8-bit unsigned integers
T_reconst_uint8 = np.clip(T_reconst, 0, 255).astype(np.uint8)

## 11. Save the Compressed Video

Write the reconstructed frames into an MP4 video. The output frame size and frame rate are taken from the original video.

In [ ]:
output_path = 'compressed_video.mp4'

if fps <= 0:
    fps = 20.0

frame_size = (vid_shape[2], vid_shape[1])

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    frame_size,
    isColor=False
)

if not output.isOpened():
    raise RuntimeError('Could not create the output video file.')

for frame in T_reconst_uint8:
    output.write(frame)

output.release()

print('Compressed video saved to:', output_path)

## 12. Compare Original and Reconstructed Frames

Display selected original and reconstructed frames side by side to visually evaluate the effect of compression.

In [ ]:
fig, axes = plt.subplots(len(frame_indices), 2, figsize=(10, 3 * len(frame_indices)))

for row, idx in enumerate(frame_indices):
    axes[row, 0].imshow(vid_tensor[idx], cmap='gray')
    axes[row, 0].set_title(f'Original - Frame {idx}')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(T_reconst_uint8[idx], cmap='gray')
    axes[row, 1].set_title(f'Reconstructed - Frame {idx}')
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()

## 13. Calculate Compression and Reconstruction Metrics

Estimate the number of scalar values required to represent the original tensor and its HOSVD approximation. The compressed representation consists of the core tensor and the three truncated factor matrices.

The storage estimate is:

\[N_{compressed} = k^3 + k(N + H + W)\]

The compression ratio is the original number of values divided by the compressed number of values. These are theoretical storage estimates and do not include video codec overhead or file-system compression.

In [ ]:
original_size = int(np.prod(vid_shape))
compressed_size = int(k ** 3 + k * sum(vid_shape))

compression_ratio = original_size / compressed_size
storage_reduction = (1 - compressed_size / original_size) * 100

mse = np.mean((frames_array - T_reconst) ** 2)
rmse = np.sqrt(mse)

if mse == 0:
    psnr = float('inf')
else:
    psnr = 10 * np.log10((255 ** 2) / mse)

print('Original number of values:', original_size)
print('Compressed number of values:', compressed_size)
print(f'Compression ratio: {compression_ratio:.2f}:1')
print(f'Theoretical storage reduction: {storage_reduction:.2f}%')
print(f'Mean Squared Error (MSE): {mse:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')
print(f'Peak Signal-to-Noise Ratio (PSNR): {psnr:.2f} dB')

## 14. Conclusion

In this experiment, a video was represented as a third-order tensor and compressed using HOSVD. SVD was applied to the unfolding matrices of the temporal and spatial modes. The resulting core tensor and truncated factor matrices were used to reconstruct the video with a reduced number of components. The compression level was controlled by k, and the reconstructed video was evaluated using storage estimates, MSE, RMSE, and PSNR.

## 15. Download the Compressed Video

Download the reconstructed MP4 video from Google Colab.

In [ ]:
files.download(output_path)